# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`
This notebook demonstrates loading, exploring, and analyzing the FAIR² dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library. It walks through accessing metadata, record sets, and conducting basic exploratory data analysis (EDA).

### Dataset Source
The dataset is accessible via a [Croissant schema](https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json).

In [ ]:
# Install mlcroissant if not already installed
!pip install mlcroissant

## 1. Data Loading
Load dataset metadata and record sets using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json"

# Load dataset metadata
dataset = mlc.Dataset(croissant_url)

# Accessing metadata as object properties
print(f"Dataset Name: {dataset.metadata.name}")
print(f"Description: {dataset.metadata.description}")


## 2. Data Overview
Review available record sets in the dataset. All entities are listed by their `@id`.

In [ ]:
# List record sets by @id
record_sets = [rs['@id'] for rs in dataset.metadata.get('recordSet', [])] if hasattr(dataset.metadata, 'get') else []
if not record_sets:
    # Try loading via mlcroissant API if not in static metadata
    try:
        # record_sets() yields dicts with @id and name attributes
        record_sets = [rs['@id'] for rs in dataset.record_sets()]
    except Exception:
        # Fallback: Try to inspect (may be empty if no direct recordSets listed)
        record_sets = []
print("Record set @ids:")
pprint.pprint(record_sets)

# Get more detail if possible
if hasattr(dataset, 'record_sets'):
    print("\nRecord Sets Detail:")
    for rs in dataset.record_sets():
        print(f"@id: {rs.get('@id', '[unknown]')}, name: {rs.get('name', '[unnamed]')}")
        # List fields for each record set
        if 'field' in rs:
            fields = rs['field'] if isinstance(rs['field'], list) else [rs['field']]
            print("  Fields @id:")
            for field in fields:
                if isinstance(field, dict):
                    print(f"    - {field.get('@id')}")
                else:
                    print(f"    - {field}")
        else:
            print("  [No fields listed]")


## 3. Data Extraction
Load a record set into a DataFrame for analysis. All entities referenced by their `@id`.

In [ ]:
# Attempt to list all record sets and their fields by @id
valid_record_sets = []
fields_per_recordset = {}
if hasattr(dataset, 'record_sets'):
    for rs in dataset.record_sets():
        rs_id = rs.get('@id', None)
        if rs_id:
            valid_record_sets.append(rs_id)
            # Get field @ids for each record set
            if 'field' in rs:
                fields = rs['field'] if isinstance(rs['field'], list) else [rs['field']]
                fields_per_recordset[rs_id] = [f['@id'] if isinstance(f, dict) and '@id' in f else str(f) for f in fields]
            else:
                fields_per_recordset[rs_id] = []

# Show what record sets were discovered
print(f"Discovered record sets: {valid_record_sets}")
print(f"Fields per record set:")
for rs_id, fields in fields_per_recordset.items():
    print(f"- {rs_id}: {fields}")

dataframes = {}
for record_set_id in valid_record_sets:
    records_iter = dataset.records(record_set=record_set_id)
    df = pd.DataFrame(records_iter)
    dataframes[record_set_id] = df
    print(f"Loaded {len(df)} records for {record_set_id}")

# Show columns for the first record set, if available
if valid_record_sets:
    first_rs = valid_record_sets[0]
    print(f"Columns in record set {first_rs}:")
    print(dataframes[first_rs].columns.tolist())
    dataframes[first_rs].head()


## 4. Exploratory Data Analysis (EDA)
Select fields by their `@id` for data processing: filter, normalize, and group.

In [ ]:
# For demonstration, we select the first record set found
if valid_record_sets:
    rs_id = valid_record_sets[0]
    df = dataframes[rs_id]
    print(f"Working on record set: {rs_id}")

    # Attempt to auto-select a numeric field (int/float) by pandas dtype
    numeric_fields = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]
    if not numeric_fields:
        # Try to convert columns to numeric if possible
        for col in df.columns:
            try:
                df[col] = pd.to_numeric(df[col], errors='ignore')
            except:
                continue
        numeric_fields = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]

    if numeric_fields:
        numeric_field_id = numeric_fields[0]
        threshold = df[numeric_field_id].mean() if pd.notnull(df[numeric_field_id].mean()) else 10
        print(f"Numeric field selected: {numeric_field_id}, threshold: {threshold:.4f}\n")
        # Filter records above threshold
        filtered_df = df[df[numeric_field_id] > threshold].copy()
        print(f"Filtered records with '{numeric_field_id}' > {threshold:.4f}:")
        print(filtered_df.head())
        # Normalize field
        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"\nNormalized '{numeric_field_id}' for filtered records:")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())
        # Group by another field (try to auto-select a likely grouping field that is not numeric)
        non_numeric_cols = [col for col in df.columns if col != numeric_field_id and not pd.api.types.is_numeric_dtype(df[col])]
        group_field = non_numeric_cols[0] if non_numeric_cols else None
        if group_field:
            grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().reset_index()
            print(f"\nGrouped data by '{group_field}':")
            print(grouped_df.head())
    else:
        print("No numeric fields found for EDA.")
else:
    print("No record sets available for data analysis.")

## 5. Visualization
Visualize distributions or relationships for selected fields.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if valid_record_sets and numeric_fields and not filtered_df.empty:
    # Plot distribution of the chosen numeric field
    plt.figure(figsize=(8,4))
    sns.histplot(filtered_df[numeric_field_id], kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.show()

    # If grouping was performed
    if group_field and not grouped_df.empty:
        plt.figure(figsize=(8,4))
        sns.barplot(x=group_field, y=numeric_field_id, data=grouped_df)
        plt.title(f"Mean {numeric_field_id} by {group_field}")
        plt.xlabel(group_field)
        plt.ylabel(f"Mean {numeric_field_id}")
        plt.xticks(rotation=45)
        plt.show()
else:
    print("Insufficient numeric/grouped data for plotting.")

## 6. Conclusion
We successfully loaded the FAIR² Croissant dataset, listed available record sets by their `@id`, and performed basic exploratory analysis on the available fields. For more detailed analysis, refer to domain documentation and the Croissant schema to map field meanings with their `@id`.